# Sliding-Window Feature Extraction — Escalation-Stage Pipeline

Foundational step for the reframed architecture (clip-level classification
-> per-window escalation-stage prediction), per Geetanjali's 2026-11-09
review: the current fusion architecture isn't novel (frozen-backbone
fusion is already published -- VadCLIP, AnomalyCLIP, CLIP-TSA all do this
on XD-Violence), and she separately asked for a live color-coded
(green/amber/red) risk indicator. Both point at the same fix: stop
classifying whole clips, start modeling risk *as it evolves over time*.

**What this notebook produces:** for riot and normal clips (both official
train and test split), a sequence of per-window embeddings instead of one
vector per clip. DINOv2 + OpenCLIP only -- the two branches confirmed to
carry real signal (`docs/findings.md`); LK/RF-DETR/VideoMAE are not
re-extracted into this pipeline given the ablation results.

**Output:** one `.npz` per clip
(`My Drive/CVPHG/sliding_window_features/<video_id>.npz`) holding window
start/end frame indices plus per-window DINOv2 and OpenCLIP embeddings,
and one manifest CSV indexing them all. Resumable -- skips any clip whose
`.npz` already exists.

**Speed changes (2026-11-16), after the first runs took ~22 s per clip:**
frames are embedded in batches instead of one at a time, each video is
read in a single forward pass without converting frames that are not
used, and the work is ordered so the clips the evaluation cannot do
without come first (all riot -> all test-split normals -> train-split
normals up to `MAX_NORMAL_TRAIN`). The embeddings themselves are computed
exactly as before, so clips already extracted stay valid and are not
redone.

**Next step:** `escalation_stage_model.ipynb` trains and evaluates on
whatever this notebook has produced so far -- it does not need the full
normal class.

## Step 0 — Setup

In [ ]:
!pip install opencv-python-headless open_clip_torch -q

In [ ]:
import csv
import os
import time

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

RIOT_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
NORMAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_normal'
OUT_DIR = '/content/drive/MyDrive/CVPHG/sliding_window_features'
os.makedirs(OUT_DIR, exist_ok=True)

# Window sizing in seconds, not frames -- clips in this dataset vary in fps
# (movie vs. YouTube sources), so a frame-count window would cover a
# different real duration per clip. A time-based window keeps "2 seconds
# of context" meaning the same thing across the whole dataset.
WINDOW_SECONDS = 2.0
STRIDE_SECONDS = 1.0  # 50% overlap between consecutive windows

# Frames are embedded in batches of this size (same numbers as one at a
# time, far fewer GPU round-trips).
EMBED_BATCH = 32

# How many official-TRAIN normal clips to extract. The escalation model
# trains on ~384 riot clips, so ~600 normals is already a 1.5:1 ratio;
# the remaining ~1,450 are the slowest clips in the dataset (long YouTube
# videos) and can be added later -- extraction is resumable. All riot
# clips and ALL test-split normal clips are always extracted, because the
# evaluation needs every one of them. Set to None for the full class.
MAX_NORMAL_TRAIN = 600

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU recommended '
          '-- this extracts many more windows per clip than the old sparse-frame pipeline.')

In [ ]:
riot_manifest = pd.read_csv(os.path.join(RIOT_DIR, 'manifest_xdviolence_riot.csv'))
riot_manifest['clip_dir'] = RIOT_DIR
normal_manifest = pd.read_csv(os.path.join(NORMAL_DIR, 'manifest_xdviolence_normal.csv'))
normal_manifest['clip_dir'] = NORMAL_DIR

all_clips = pd.concat([riot_manifest, normal_manifest], ignore_index=True)
print(f"{len(riot_manifest)} riot + {len(normal_manifest)} normal = {len(all_clips)} clips total")
print(all_clips.groupby(['label', 'split']).size())

## Step 1 — Load DINOv2 and OpenCLIP (frozen, no fine-tuning)

In [ ]:
dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2_model.eval().to(DEVICE)

DINOV2_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DINOV2_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DINOV2_DIM = 768


def dinov2_embed_batch(frames_rgb) -> np.ndarray:
    """Same preprocessing as the original one-frame-at-a-time version
    (224x224 INTER_AREA resize, ImageNet normalisation), just stacked."""
    imgs = np.stack([cv2.resize(f, (224, 224), interpolation=cv2.INTER_AREA) for f in frames_rgb])
    tensor = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0
    tensor = ((tensor - DINOV2_MEAN) / DINOV2_STD).to(DEVICE)
    with torch.no_grad():
        feat = dinov2_model(tensor)
    return feat.cpu().numpy()

In [ ]:
import open_clip
from PIL import Image

openclip_model, _, openclip_preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
openclip_model.eval().to(DEVICE)
OPENCLIP_DIM = 768


def openclip_embed_batch(frames_rgb) -> np.ndarray:
    """Same preprocessing as the original one-frame-at-a-time version, stacked."""
    tensor = torch.stack([openclip_preprocess(Image.fromarray(f)) for f in frames_rgb]).to(DEVICE)
    with torch.no_grad():
        feat = openclip_model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()

## Step 2 — Window the clip and embed the center frame of each window

One representative frame per window (the center frame) is embedded, not
every frame -- this keeps extraction cost comparable to the earlier
sparse-sampling pipeline while still producing a real sequence across the
clip, instead of one pooled vector for the whole thing.

In [ ]:
def compute_windows(fps, total_frames):
    """Returns a list of (start_frame, end_frame, center_frame) for
    windows of WINDOW_SECONDS, stepped every STRIDE_SECONDS, covering the
    full clip duration."""
    if fps <= 0 or total_frames <= 0:
        return []

    window_frames = max(1, int(round(WINDOW_SECONDS * fps)))
    stride_frames = max(1, int(round(STRIDE_SECONDS * fps)))

    windows = []
    start = 0
    while start < total_frames:
        end = min(start + window_frames, total_frames)
        center = (start + end) // 2
        windows.append((start, end, center))
        if end >= total_frames:
            break
        start += stride_frames
    return windows


def iter_center_frames(video_path, center_frames):
    """Opens the video ONCE and walks it forward, yielding
    (window_index, rgb_frame) for each requested frame as it is reached.

    - No reopening and no seeking: over a Drive-mounted path every
      seek was a fresh network round-trip plus a decode from the nearest
      keyframe, which is what made the first version take minutes per clip.
    - grab() advances without converting the frame; only the frames that
      are actually embedded pay for retrieve() + colour conversion.
    - Frames are yielded one at a time, so a long video never has all of
      its sampled frames in memory at once."""
    wanted = {f: i for i, f in enumerate(center_frames)}
    last = center_frames[-1]
    cap = cv2.VideoCapture(video_path)
    try:
        idx = 0
        while idx <= last:
            if not cap.grab():
                break
            if idx in wanted:
                ok, frame_bgr = cap.retrieve()
                if ok:
                    yield wanted[idx], cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            idx += 1
    finally:
        cap.release()


def extract_clip(video_path):
    """Returns the arrays saved per clip, or None if the file is unreadable.
    A window whose frame could not be decoded keeps all-zero rows and
    start == end == 0, which downstream code treats as 'missing'."""
    probe = cv2.VideoCapture(video_path)
    fps = probe.get(cv2.CAP_PROP_FPS) or 0.0
    total_frames = int(probe.get(cv2.CAP_PROP_FRAME_COUNT))
    probe.release()

    windows = compute_windows(fps, total_frames)
    if not windows:
        return None

    n = len(windows)
    out = dict(
        window_start=np.zeros(n, dtype=np.int32), window_end=np.zeros(n, dtype=np.int32),
        dinov2=np.zeros((n, DINOV2_DIM), dtype=np.float32),
        openclip=np.zeros((n, OPENCLIP_DIM), dtype=np.float32),
        fps=fps, total_frames=total_frames,
    )
    batch_w, batch_f = [], []

    def flush():
        if not batch_f:
            return
        out['dinov2'][batch_w] = dinov2_embed_batch(batch_f)
        out['openclip'][batch_w] = openclip_embed_batch(batch_f)
        for w in batch_w:
            out['window_start'][w], out['window_end'][w] = windows[w][0], windows[w][1]
        batch_w.clear()
        batch_f.clear()

    for w, frame in iter_center_frames(video_path, [c for (_, _, c) in windows]):
        batch_w.append(w)
        batch_f.append(frame)
        if len(batch_f) == EMBED_BATCH:
            flush()
    flush()
    return out

## Step 3 — Per-clip extraction loop (resumable)

In [ ]:
already_done = {f[:-4] for f in os.listdir(OUT_DIR) if f.endswith('.npz') and not f.endswith('.tmp.npz')}
print(f"Resuming: {len(already_done)} clips already have window features.")

# Order of work: every riot clip, then every TEST-split normal clip (the
# evaluation cannot run without them), then train-split normals up to
# MAX_NORMAL_TRAIN. Clips already extracted always count and are kept.
# The normal manifest is already a seeded random shuffle, so taking the
# next ones in manifest order is an unbiased sample.
is_done = all_clips['id'].isin(already_done)
riot_q = all_clips[all_clips['label'] == 'riot']
norm_test_q = all_clips[(all_clips['label'] == 'normal') & (all_clips['split'] == 'test')]
norm_train = all_clips[(all_clips['label'] == 'normal') & (all_clips['split'] == 'train')]
if MAX_NORMAL_TRAIN is not None:
    done_train = norm_train[is_done.loc[norm_train.index]]
    pending_train = norm_train[~is_done.loc[norm_train.index]]
    norm_train = pd.concat([done_train, pending_train.head(max(0, MAX_NORMAL_TRAIN - len(done_train)))])
queue = pd.concat([riot_q, norm_test_q, norm_train], ignore_index=True)
n_pending = int((~queue['id'].isin(already_done)).sum())
print(f"Plan: {len(riot_q)} riot + {len(norm_test_q)} test normals + {len(norm_train)} train normals "
      f"= {len(queue)} clips; {n_pending} still to extract.")

manifest_rows = []
n_new, t0 = 0, time.time()
for row in queue.to_dict('records'):
    video_id = row['id']
    out_path = os.path.join(OUT_DIR, f"{video_id}.npz")

    if video_id not in already_done:
        video_path = os.path.join(row['clip_dir'], video_id)
        try:
            arrays = extract_clip(video_path)
            if arrays is None:
                print(f"  SKIPPED {video_id}: unreadable or zero-length")
                continue
            # Write to a temp name, then rename: an interrupted run never
            # leaves a half-written file that a later resume would trust.
            tmp_path = out_path + '.tmp.npz'
            np.savez_compressed(tmp_path, **arrays)
            os.replace(tmp_path, out_path)
        except Exception as e:
            print(f"  FAILED {video_id}: {e}")
            continue
        n_new += 1
        if n_new % 25 == 0 or n_new == n_pending:
            per_clip = (time.time() - t0) / n_new
            print(f"  {n_new}/{n_pending} new clips done -- {per_clip:.1f}s per clip, "
                  f"~{(n_pending - n_new) * per_clip / 60:.0f} min left")

    manifest_rows.append({'video_id': video_id, 'label': row['label'], 'split': row['split']})

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(OUT_DIR, 'manifest_sliding_window.csv')
manifest_df.to_csv(manifest_path, index=False)
print(f"\nDone. {len(manifest_df)}/{len(queue)} planned clips have window features.")
print(manifest_df.groupby(['label', 'split']).size())
print(f"Manifest -> {manifest_path}")

## Sanity check — window counts look right?

A handful of clips at a few different durations, to confirm the window
count scales the way it should before trusting this for the full run.

In [ ]:
sample_ids = manifest_df['video_id'].sample(min(5, len(manifest_df)), random_state=42).tolist()
for vid in sample_ids:
    data = np.load(os.path.join(OUT_DIR, f"{vid}.npz"))
    duration_s = data['total_frames'] / data['fps'] if data['fps'] > 0 else 0
    print(f"{vid}: {duration_s:.1f}s clip -> {len(data['window_start'])} windows "
          f"(fps={float(data['fps']):.1f})")

## Next steps

- **Train and evaluate:** `escalation_stage_model.ipynb` -- the ordinal
  stage filter, the clip-referenced escalation pathway, the baseline and
  ablation arms, the lead-time evaluation against `annotations.txt`, and
  the pre-onset confound check are all in that notebook. It runs on a
  partial extraction and prints which clips it had.
- **More normals later:** raise `MAX_NORMAL_TRAIN` (or set it to `None`)
  and re-run this notebook; only the missing clips are extracted. Then
  re-run the model notebook and compare against the smaller-set result.